# Biohub: Learned U-Net + Transformer + ILP Exact Validation

**Question:** does the genuinely learned/global pipeline in
`cellmot-baseline-artifacts` beat the best classical exact-validation score
(`0.8104577161`) on the same embryo-disjoint samples?

This is a validation-only gate. It runs the published pretrained temporal 3D
U-Net detector, learned edge transformer, and ILP optimizer. No test inference or
competition submission is produced.

**Success criterion:** exact aggregate score above `0.8104577161`, with the
per-embryo results and division counts inspected before any tuning.


## Plan and controls

1. Attach the public `cellmot-baseline-artifacts` dataset.
2. Install Zarr, tracksdata, and PySCIPOpt into an isolated target directory.
3. Copy the artifact repository and pretrained weights into `/kaggle/working`.
4. Run the published default learned pipeline (`det_threshold=0.99`,
   `division_weight=1.0`) on one video per embryo.
5. Score predicted GEFF graphs with the exact official evaluator in a subprocess.

The first run deliberately avoids the public notebook's proxy sweep. We need a
trustworthy default reproduction before spending GPU time on tuning.


In [ ]:
from __future__ import annotations

import json
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch

COMPETITION = "biohub-cell-tracking-during-development"
METHOD = "unet_transformer"
SEED = 42
np.random.seed(SEED)

DET_THRESHOLD = 0.99
ILP_EDGE_WEIGHT = -1.0
ILP_APPEARANCE_WEIGHT = 0.1
ILP_DISAPPEARANCE_WEIGHT = 0.1
ILP_DIVISION_WEIGHT = 1.0
UNET_BATCH_SIZE = 4
MAX_VALIDATION_EMBRYOS = 3
CLASSICAL_BENCHMARK = 0.8104577160678552

COMP_DIR = next(
    (path for path in [
        Path(f"/kaggle/input/competitions/{COMPETITION}"),
        Path(f"/kaggle/input/{COMPETITION}"),
    ] if path.exists()),
    Path(f"/kaggle/input/competitions/{COMPETITION}"),
)
TRAIN_DIR = COMP_DIR / "train"
WORKING = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
REPO_DIR = WORKING / "cellmot_repo"
LEARNED_SITE = WORKING / "cellmot_learned_site"
LEARNED_MARKER = LEARNED_SITE / ".install_complete"

print("Competition:", COMP_DIR)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: this learned validation is intended for a GPU kernel")


## Offline artifact and isolated dependencies

The notebook has internet disabled. All repository code, wheels, and weights come
from the attached public artifact dataset. Dependency wheels are installed under
`cellmot_learned_site` and exposed only to subprocesses through `PYTHONPATH`.


In [ ]:
def find_artifacts() -> Path:
    candidates = [
        Path("/kaggle/input/datasets/thibautgoldsborough/"
             "cellmot-baseline-artifacts/cellmot-baseline-artifacts"),
        Path("/kaggle/input/cellmot-baseline-artifacts/cellmot-baseline-artifacts"),
        Path("/kaggle/input/cellmot-baseline-artifacts"),
    ]
    root = Path("/kaggle/input")
    if root.exists():
        for child in root.iterdir():
            if "cellmot" in child.name.lower():
                candidates.extend([child, child / child.name])
    for candidate in candidates:
        if ((candidate / "repo").exists()
                and (candidate / "weights").exists()
                and (candidate / "wheels").exists()):
            return candidate
    raise FileNotFoundError("Attach thibautgoldsborough/cellmot-baseline-artifacts")


ARTIFACTS = find_artifacts()
if not LEARNED_MARKER.exists():
    LEARNED_SITE.mkdir(exist_ok=True)
    subprocess.run([
        sys.executable, "-m", "pip", "install", "--quiet", "--no-cache-dir",
        "--ignore-installed", "--target", str(LEARNED_SITE),
        "--no-index", "--find-links", str(ARTIFACTS / "wheels"),
        "tracksdata", "zarr>=3.0.10", "pyscipopt",
    ], check=True)
    LEARNED_MARKER.write_text("ok\n")

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
shutil.copytree(ARTIFACTS / "repo", REPO_DIR)
shutil.copytree(ARTIFACTS / "weights", REPO_DIR / "weights", dirs_exist_ok=True)

LEARNED_ENV = {
    **os.environ,
    "PYTHONPATH": os.pathsep.join([str(LEARNED_SITE), str(REPO_DIR / "src")]),
    "PYTHONNOUSERSITE": "1",
}

weights_root = REPO_DIR / "weights" / METHOD
available_splits = sorted(path.name for path in weights_root.iterdir() if path.is_dir())
if not available_splits:
    raise FileNotFoundError(f"No learned checkpoints under {weights_root}")
SPLIT_NAME = "split_0" if "split_0" in available_splits else available_splits[0]
WEIGHTS = weights_root / SPLIT_NAME / "edge_predictor_best.pth"
if not WEIGHTS.exists():
    raise FileNotFoundError(WEIGHTS)

environment_check = subprocess.run([
    sys.executable, "-c",
    "import numpy, torch, zarr, tracksdata, pyscipopt; "
    "print(numpy.__version__, torch.__version__, torch.cuda.is_available())",
], env=LEARNED_ENV, capture_output=True, text=True, check=True)

print("Artifacts:", ARTIFACTS)
print("Available splits:", available_splits)
print("Selected checkpoint:", WEIGHTS.relative_to(REPO_DIR))
print("Isolated environment:", environment_check.stdout.strip())


## Exact evaluator bridge

Both prediction and evaluation operate on GEFF directories inside the isolated
subprocess environment. The runner loads predicted and truth graphs, invokes the
official metric, and returns node/edge/division statistics as JSON.


In [ ]:
METRIC_RUNNER = WORKING / "learned_exact_metric_runner.py"
METRIC_RUNNER.write_text(r"""
import json
import sys
from pathlib import Path

import numpy as np
import polars as pl
import tracksdata as td
import zarr

repo_src, prediction_path, truth_path, output_path = sys.argv[1:]
sys.path.insert(0, repo_src)
from tracking_cellmot.metrics import evaluate, node_recall, per_sample_metrics

K = td.DEFAULT_ATTR_KEYS


def load_arrays(path):
    group = zarr.open(str(path), mode="r")
    return (
        np.asarray(group["nodes/ids"][:], dtype=np.int64),
        np.asarray(group["nodes/props/t/values"][:], dtype=np.int64),
        np.asarray(group["nodes/props/z/values"][:], dtype=np.float64),
        np.asarray(group["nodes/props/y/values"][:], dtype=np.float64),
        np.asarray(group["nodes/props/x/values"][:], dtype=np.float64),
        np.asarray(group["edges/ids"][:], dtype=np.int64).reshape(-1, 2),
    )


def build_graph(arrays):
    node_ids, node_t, node_z, node_y, node_x, edges = arrays
    graph = td.graph.InMemoryGraph()
    for key in (K.Z, K.Y, K.X):
        graph.add_node_attr_key(key, pl.Float64, 0.0)
    node_map = {}
    for index, node_id in enumerate(node_ids):
        node_map[int(node_id)] = graph.add_node({
            K.T: int(node_t[index]), K.Z: float(node_z[index]),
            K.Y: float(node_y[index]), K.X: float(node_x[index]),
        })
    for source_id, target_id in edges:
        graph.add_edge(node_map[int(source_id)], node_map[int(target_id)], {})
    return graph


def find_estimated_nodes(path):
    try:
        metadata = json.loads((Path(path) / "zarr.json").read_text())
    except Exception:
        return float("nan")
    def search(value):
        if isinstance(value, dict):
            if "estimated_number_of_nodes" in value:
                return value["estimated_number_of_nodes"]
            for child in value.values():
                result = search(child)
                if result is not None:
                    return result
        return None
    result = search(metadata)
    return float(result) if result is not None else float("nan")


prediction_arrays = load_arrays(prediction_path)
truth_arrays = load_arrays(truth_path)
prediction = build_graph(prediction_arrays)
truth = build_graph(truth_arrays)
estimated_nodes = find_estimated_nodes(truth_path)

counts = evaluate(
    prediction, truth, scale=(1.625, 0.40625, 0.40625), max_distance=7.0,
)
recall = node_recall(prediction, truth)
metrics = per_sample_metrics(counts, estimated_nodes, recall)
pred_edges = prediction_arrays[-1]
if len(pred_edges):
    _, out_counts = np.unique(pred_edges[:, 0], return_counts=True)
    predicted_divisions = int((out_counts >= 2).sum())
else:
    predicted_divisions = 0
metrics.update({
    "estimated_nodes": estimated_nodes,
    "num_pred_edges": int(len(pred_edges)),
    "num_pred_divisions": predicted_divisions,
})
Path(output_path).write_text(json.dumps(metrics, default=lambda value: value.item()))
""")


def evaluate_geff(prediction_path: Path, truth_path: Path, dataset: str) -> dict:
    output_path = WORKING / f"{dataset}_learned_metrics.json"
    result = subprocess.run([
        sys.executable, str(METRIC_RUNNER), str(REPO_DIR / "src"),
        str(prediction_path), str(truth_path), str(output_path),
    ], env=LEARNED_ENV, capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError("Exact metric failed:\n" + result.stderr[-4000:])
    row = json.loads(output_path.read_text())
    row.update(dataset=dataset, embryo=dataset.split("_")[0])
    return row


print("Exact GEFF evaluator ready")


## Embryo-disjoint validation samples

Use the same deterministic one-video-per-embryo selection as the classical
benchmark. This competition currently exposes two training embryo prefixes.


In [ ]:
def list_zarr_names(directory: Path) -> list[str]:
    return sorted(path.stem for path in directory.glob("*.zarr"))


def select_validation_samples() -> list[str]:
    selected, seen = [], set()
    for name in list_zarr_names(TRAIN_DIR):
        embryo = name.split("_")[0]
        if embryo in seen or not (TRAIN_DIR / f"{name}.geff").exists():
            continue
        selected.append(name)
        seen.add(embryo)
        if len(selected) >= MAX_VALIDATION_EMBRYOS:
            break
    return selected


validation_samples = select_validation_samples()
if not validation_samples:
    raise RuntimeError("No labeled training samples found")
print("Validation samples:", validation_samples)


## Learned prediction

The repository script performs neural detection, learned edge scoring, and ILP
tracking. Its stdout/stderr are saved for auditability. Predictions from prior
runs are deleted before execution to prevent cross-run contamination.


In [ ]:
def run_learned_prediction(sample_names: list[str]) -> tuple[list[Path], float]:
    splits_file = REPO_DIR / "splits_exact_validation.json"
    splits_file.write_text(json.dumps([{"split": 0, "train": [], "test": sample_names}]))

    prediction_root = REPO_DIR / "predictions"
    if prediction_root.exists():
        shutil.rmtree(prediction_root)

    command = [
        sys.executable, "scripts/predict_unet_transformer.py",
        "--data-dir", str(TRAIN_DIR),
        "--splits", splits_file.name,
        "--split", "0",
        "--weights", str(WEIGHTS.relative_to(REPO_DIR)),
        "--unet-batch-size", str(UNET_BATCH_SIZE),
        "--det-threshold", str(DET_THRESHOLD),
        "--ilp-edge-weight", str(ILP_EDGE_WEIGHT),
        "--ilp-appearance-weight", str(ILP_APPEARANCE_WEIGHT),
        "--ilp-disappearance-weight", str(ILP_DISAPPEARANCE_WEIGHT),
        "--ilp-division-weight", str(ILP_DIVISION_WEIGHT),
        "--use-ilp",
    ]
    started = time.time()
    result = subprocess.run(
        command, cwd=REPO_DIR, env=LEARNED_ENV,
        capture_output=True, text=True,
    )
    runtime_seconds = time.time() - started
    (WORKING / "learned_predict.stdout.log").write_text(result.stdout)
    (WORKING / "learned_predict.stderr.log").write_text(result.stderr)
    if result.returncode != 0:
        raise RuntimeError(
            f"Learned prediction failed with code {result.returncode}:\n"
            + result.stderr[-5000:]
        )

    predictions = sorted(prediction_root.rglob("*.geff"))
    prediction_by_name = {path.stem: path for path in predictions}
    missing = sorted(set(sample_names) - set(prediction_by_name))
    if missing:
        raise RuntimeError(f"Missing learned predictions: {missing}")
    return [prediction_by_name[name] for name in sample_names], runtime_seconds


print("Prediction command configured for", SPLIT_NAME)


## Run and exact decision gate

This is the expensive cell. The result is compared directly with the NMS-3.8
classical benchmark; no proxy metric is used.


In [ ]:
prediction_paths, prediction_runtime = run_learned_prediction(validation_samples)

validation_rows = []
for index, (name, prediction_path) in enumerate(zip(validation_samples, prediction_paths), start=1):
    started = time.time()
    row = evaluate_geff(prediction_path, TRAIN_DIR / f"{name}.geff", name)
    row["metric_runtime_seconds"] = round(time.time() - started, 2)
    validation_rows.append(row)
    print(
        f"[{index}/{len(validation_samples)}] {name}: "
        f"adj={row['adj_edge_jaccard']:.6f} "
        f"edge={row['edge_tp']}/{row['edge_fp']}/{row['edge_fn']} "
        f"nodes={row['num_pred_nodes']} divisions={row['num_pred_divisions']}"
    )

validation_df = pd.DataFrame(validation_rows)
weights = validation_df["edge_tp"] + validation_df["edge_fp"] + validation_df["edge_fn"]
aggregate_edge = float(
    (validation_df["adj_edge_jaccard"] * weights).sum() / max(float(weights.sum()), 1.0)
)
division_tp = int(validation_df["division_tp"].sum())
division_fp = int(validation_df["division_fp"].sum())
division_fn = int(validation_df["division_fn"].sum())
division_denominator = division_tp + division_fp + division_fn
division_jaccard = (
    division_tp / division_denominator if division_denominator else float("nan")
)
exact_score = (
    aggregate_edge if not np.isfinite(division_jaccard)
    else aggregate_edge + 0.1 * division_jaccard
)
summary = {
    "method": METHOD,
    "split": SPLIT_NAME,
    "det_threshold": DET_THRESHOLD,
    "ilp_division_weight": ILP_DIVISION_WEIGHT,
    "n_samples": len(validation_df),
    "adj_edge_jaccard": aggregate_edge,
    "division_jaccard": division_jaccard,
    "score": exact_score,
    "classical_benchmark": CLASSICAL_BENCHMARK,
    "delta_vs_classical": exact_score - CLASSICAL_BENCHMARK,
    "beats_classical": bool(exact_score > CLASSICAL_BENCHMARK),
    "edge_tp": int(validation_df["edge_tp"].sum()),
    "edge_fp": int(validation_df["edge_fp"].sum()),
    "edge_fn": int(validation_df["edge_fn"].sum()),
    "division_tp": division_tp,
    "division_fp": division_fp,
    "division_fn": division_fn,
    "prediction_runtime_seconds": round(prediction_runtime, 2),
}

display(validation_df[[
    "dataset", "embryo", "node_recall", "total_node_ratio",
    "num_pred_nodes", "num_pred_edges", "num_pred_divisions",
    "edge_tp", "edge_fp", "edge_fn", "adj_edge_jaccard",
    "division_tp", "division_fp", "division_fn", "metric_runtime_seconds",
]])
print("Exact learned summary:", json.dumps(summary, indent=2))

validation_df.to_csv(WORKING / "learned_validation_metrics.csv", index=False)
(WORKING / "learned_validation_summary.json").write_text(
    json.dumps(summary, indent=2)
)
(WORKING / "learned_run_manifest.json").write_text(json.dumps({
    "validation_samples": validation_samples,
    "checkpoint": str(WEIGHTS.relative_to(REPO_DIR)),
    "parameters": {
        "det_threshold": DET_THRESHOLD,
        "unet_batch_size": UNET_BATCH_SIZE,
        "ilp_edge_weight": ILP_EDGE_WEIGHT,
        "ilp_appearance_weight": ILP_APPEARANCE_WEIGHT,
        "ilp_disappearance_weight": ILP_DISAPPEARANCE_WEIGHT,
        "ilp_division_weight": ILP_DIVISION_WEIGHT,
    },
}, indent=2))


## Decision rule

- If exact learned score is materially above `0.810458`, tune detection threshold
  and division weight with the exact evaluator, then create a separate test kernel.
- If it is similar or worse, do not spend a submission; inspect edge and division
  failure modes before changing parameters.
- This notebook never writes `submission.csv` and cannot be submitted by mistake.
